In [1]:
import numpy as np

# ─────────────────────────────────────────
# SCENARIO SETUP
# ─────────────────────────────────────────
# Each scenario defines outcomes and their joint probabilities.
# x_vals, y_vals: the possible outcome values
# p_vals: the joint probability p_ij for each (x_i, y_j) pair

scenarios = {
    "Independent Coins": {
        "x_vals": [0, 0, 1, 1],
        "y_vals": [0, 1, 0, 1],
        "p_vals": [0.25, 0.25, 0.25, 0.25],
    },
    "Glued Coins": {
        "x_vals": [0, 1],
        "y_vals": [0, 1],
        "p_vals": [0.5, 0.5],
    },
}

# ─────────────────────────────────────────
# MATRIX COVARIANCE CALCULATION
# ─────────────────────────────────────────
# For each outcome k with probability p_k:
#   - Build vector X_k = [x_k, y_k]  (2x1 column vector)
#   - Compute mean vector m = sum_k p_k * X_k
#   - Center: U_k = X_k - m
#   - Covariance matrix: V = sum_k p_k * (U_k @ U_k.T)
#     This is the matrix version of E[UU^T]

for name, sc in scenarios.items():
    x_vals = np.array(sc["x_vals"], dtype=float)
    y_vals = np.array(sc["y_vals"], dtype=float)
    p_vals = np.array(sc["p_vals"], dtype=float)
    n = len(p_vals)

    # Step 1 — Build outcome vectors X_k as columns of a matrix (2 x n)
    X = np.vstack([x_vals, y_vals])          # shape (2, n)
    print(f"\n{'='*50}")
    print(f"SCENARIO: {name}")
    print(f"{'='*50}")
    print(f"\nStep 1 — Outcome matrix X (each column is one [x,y] pair):\n{X}")
    print(f"Joint probabilities p: {p_vals}")

    # Step 2 — Compute marginals p_i and p_j
    # p_i: probability of each unique x value
    unique_x = np.unique(x_vals)
    p_i = {xi: sum(p_vals[x_vals == xi]) for xi in unique_x}
    unique_y = np.unique(y_vals)
    p_j = {yj: sum(p_vals[y_vals == yj]) for yj in unique_y}
    print(f"\nStep 2 — Marginals:")
    print(f"  p_i (row marginals): { {k: round(v,4) for k,v in p_i.items()} }")
    print(f"  p_j (col marginals): { {k: round(v,4) for k,v in p_j.items()} }")

    # Step 3 — Compute mean vector m = [m1, m2]
    m = X @ p_vals                           # shape (2,)
    m1, m2 = m
    print(f"\nStep 3 — Mean vector m = X @ p:")
    print(f"  m1 = E[x] = {m1}")
    print(f"  m2 = E[y] = {m2}")

    # Step 4 — Center the outcome vectors: U_k = X_k - m
    U = X - m.reshape(2, 1)                  # shape (2, n), broadcast subtract
    print(f"\nStep 4 — Centered matrix U = X - m (each column is U_k):\n{U}")

    # Step 5 — Covariance matrix V = sum_k p_k * (U_k @ U_k.T)
    # This is equivalent to U @ diag(p) @ U.T
    V = U @ np.diag(p_vals) @ U.T            # shape (2, 2)
    print(f"\nStep 5 — Covariance matrix V = U @ diag(p) @ U.T:")
    print(f"{V}")
    print(f"\n  Var(x)       = V[0,0] = {V[0,0]}")
    print(f"  Var(y)       = V[1,1] = {V[1,1]}")
    print(f"  Cov(x,y)     = V[0,1] = {V[0,1]}")

    # Step 6 — Correlation matrix
    std = np.sqrt(np.diag(V))
    corr = V / np.outer(std, std)
    print(f"\nStep 6 — Correlation matrix rho = V / (std_x * std_y):")
    print(f"{np.round(corr, 4)}")


SCENARIO: Independent Coins

Step 1 — Outcome matrix X (each column is one [x,y] pair):
[[0. 0. 1. 1.]
 [0. 1. 0. 1.]]
Joint probabilities p: [0.25 0.25 0.25 0.25]

Step 2 — Marginals:
  p_i (row marginals): {np.float64(0.0): np.float64(0.5), np.float64(1.0): np.float64(0.5)}
  p_j (col marginals): {np.float64(0.0): np.float64(0.5), np.float64(1.0): np.float64(0.5)}

Step 3 — Mean vector m = X @ p:
  m1 = E[x] = 0.5
  m2 = E[y] = 0.5

Step 4 — Centered matrix U = X - m (each column is U_k):
[[-0.5 -0.5  0.5  0.5]
 [-0.5  0.5 -0.5  0.5]]

Step 5 — Covariance matrix V = U @ diag(p) @ U.T:
[[0.25 0.  ]
 [0.   0.25]]

  Var(x)       = V[0,0] = 0.25
  Var(y)       = V[1,1] = 0.25
  Cov(x,y)     = V[0,1] = 0.0

Step 6 — Correlation matrix rho = V / (std_x * std_y):
[[1. 0.]
 [0. 1.]]

SCENARIO: Glued Coins

Step 1 — Outcome matrix X (each column is one [x,y] pair):
[[0. 1.]
 [0. 1.]]
Joint probabilities p: [0.5 0.5]

Step 2 — Marginals:
  p_i (row marginals): {np.float64(0.0): np.float64(0.